# Audio Noise Remover from Video 🎬🔊

**A Python application that extracts the audio from a video, removes background noise, and re-embeds the cleaned audio back into the video.**

| | |
|---|---|
| **Developed By** | Alok Sahni |
| **Course** | NIELIT 'O' Level Examination |
| **Institute** | C Institute, Bikaner, Rajasthan |
| **Guide** | Mr. Tarun Verma (A Level NIELIT, M.Sc. Computer Science, MGSU Bikaner) |
| **Language** | Python 3.x |
| **Tools** | Tkinter, MoviePy, SciPy, Noisereduce, FFmpeg |

## 1. Problem Statement

In many real-life situations such as interviews, podcasts, classroom recordings, or webinars, audio embedded in videos often contains background noise (hiss, static, ambient hum). This affects the clarity and professionalism of the content.

**Objective:** extract the audio from a video file, remove background noise, and re-integrate the cleaned audio into the original video — using Python programming.

## 2. Objectives

- Build a GUI-based application in Python to remove noise from audio embedded in videos
- Extract audio from `.mp4` or `.mkv` files
- Reduce noise using signal processing (spectral gating)
- Save and play both original and cleaned audio for comparison
- Merge cleaned audio back into the video using FFmpeg
- Demonstrate the use of external libraries and system utilities in a Python project

## 3. Tools Used

| Tool/Library | Purpose |
|---|---|
| Python 3.x | Programming Language |
| Tkinter | GUI Interface |
| moviepy | Extracting audio from video |
| scipy.io.wavfile | Reading/writing audio in WAV format |
| noisereduce | Reducing noise from audio |
| playsound | Audio preview |
| tempfile | Handling temporary files |
| subprocess | Running FFmpeg commands |
| FFmpeg | Replacing audio in video |

## 4. Setup — Install Dependencies

Run the cell below once. On **Google Colab / Kaggle**, FFmpeg is already pre-installed. On a local machine, install FFmpeg separately from [ffmpeg.org](https://ffmpeg.org/download.html).

In [1]:
# Install required libraries
!pip install moviepy==1.0.3 noisereduce scipy playsound==1.2.2 --quiet

# Verify FFmpeg is available
!ffmpeg -version | head -1

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers


## 5. Notebook Version — Core Noise-Removal Pipeline (runs anywhere)

Jupyter notebooks (especially Colab/Kaggle) cannot display a Tkinter desktop window, so this section implements the **same processing pipeline as a plain function** that runs in any notebook:

```
video ──► extract audio (MoviePy) ──► stereo→mono (SciPy)
      ──► noise profile = first 5000 samples ──► noisereduce
      ──► save 16-bit WAV ──► FFmpeg merge ──► <name>_cleaned.mp4
```

In [2]:
import os
import shutil
import tempfile
import subprocess
from moviepy.editor import VideoFileClip
from scipy.io import wavfile
import noisereduce as nr


def clean_video(video_path, ffmpeg_path=None, noise_samples=5000):
    """Remove background noise from a video's audio track.

    Parameters
    ----------
    video_path : str
        Path to the input .mp4 / .mkv file.
    ffmpeg_path : str, optional
        Path to the ffmpeg executable. Auto-detected if not given.
    noise_samples : int
        Number of initial samples used as the noise profile
        (the original project uses the first 5000 samples).

    Returns
    -------
    output_video_path : str
        Path of the cleaned video (<name>_cleaned.mp4).
    cleaned_audio_path : str
        Path of the cleaned audio WAV file.
    """
    if not os.path.exists(video_path):
        raise FileNotFoundError(video_path)

    # Auto-detect ffmpeg (works on Colab/Kaggle/Linux); fall back to
    # the Windows path used in the original desktop application.
    if ffmpeg_path is None:
        ffmpeg_path = shutil.which("ffmpeg") or r"C:\ffmpeg\bin\ffmpeg.exe"

    # 1. Extract audio from the video into a temporary WAV file
    print("Step 1/5  Extracting audio ...")
    clip = VideoFileClip(video_path)
    temp_audio = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
    temp_audio_path = temp_audio.name
    temp_audio.close()
    clip.audio.write_audiofile(temp_audio_path, verbose=False, logger=None)
    clip.close()

    # 2. Read WAV; convert stereo to mono if needed
    print("Step 2/5  Reading WAV / converting to mono ...")
    rate, data = wavfile.read(temp_audio_path)
    if len(data.shape) == 2:
        data = data.mean(axis=1)

    # 3. Take the noise profile from the start of the recording
    noise_sample = data[:noise_samples]

    # 4. Reduce noise (spectral gating)
    print("Step 3/5  Reducing noise ...")
    reduced_noise = nr.reduce_noise(y=data, sr=rate, y_noise=noise_sample)

    # 5. Save cleaned audio as 16-bit WAV
    print("Step 4/5  Saving cleaned audio ...")
    cleaned_audio_file = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
    wavfile.write(cleaned_audio_file.name, rate, reduced_noise.astype("int16"))
    cleaned_audio_path = cleaned_audio_file.name

    # 6. Re-merge cleaned audio into the video via FFmpeg
    print("Step 5/5  Merging cleaned audio back into the video ...")
    output_video_path = os.path.splitext(video_path)[0] + "_cleaned.mp4"
    command = [
        ffmpeg_path, "-y",
        "-i", video_path,
        "-i", cleaned_audio_path,
        "-c:v", "copy",
        "-map", "0:v:0",
        "-map", "1:a:0",
        output_video_path,
    ]
    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode != 0:
        print("FFmpeg Error:", result.stderr[-500:])

    # Clean up the temporary extracted-audio file
    if os.path.exists(temp_audio_path):
        os.unlink(temp_audio_path)

    print("Done! Cleaned video saved to:", output_video_path)
    return output_video_path, cleaned_audio_path

/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':



### Run the pipeline on a video

Upload a video to the notebook environment (in Colab: folder icon → upload), set its name below, and run.

In [3]:
# Set your input video file here
INPUT_VIDEO = "sample.mp4"   # <-- change to your file name

if os.path.exists(INPUT_VIDEO):
    output_video, cleaned_wav = clean_video(INPUT_VIDEO)
else:
    print(f"'{INPUT_VIDEO}' not found — upload a video and update INPUT_VIDEO.")

'sample.mp4' not found — upload a video and update INPUT_VIDEO.


### Listen to the result (before vs after)

The players below let you compare the original and cleaned audio directly in the notebook — the notebook equivalent of the desktop app's two *Preview* buttons.

In [4]:
from IPython.display import Audio, display

if os.path.exists(INPUT_VIDEO):
    # Original audio
    clip = VideoFileClip(INPUT_VIDEO)
    orig_wav = "original_preview.wav"
    clip.audio.write_audiofile(orig_wav, verbose=False, logger=None)
    clip.close()

    print("Original (noisy) audio:")
    display(Audio(orig_wav))

    print("Cleaned audio:")
    display(Audio(cleaned_wav))

### Optional: waveform comparison

A quick visual before/after of the audio signal.

In [5]:
import matplotlib.pyplot as plt

if os.path.exists(INPUT_VIDEO):
    rate_o, data_o = wavfile.read(orig_wav)
    if len(data_o.shape) == 2:
        data_o = data_o.mean(axis=1)
    rate_c, data_c = wavfile.read(cleaned_wav)

    fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
    axes[0].plot(data_o, linewidth=0.3)
    axes[0].set_title("Original audio (noisy)")
    axes[1].plot(data_c, linewidth=0.3, color="green")
    axes[1].set_title("Cleaned audio (noise reduced)")
    axes[1].set_xlabel("Sample index")
    plt.tight_layout()
    plt.show()

## 6. Original Desktop Application — Full Tkinter GUI Source Code

This is the **complete source code of the desktop application** as submitted in the project report. It opens a 500×400 window with Browse, Clean Audio, and two Preview buttons.

> ⚠️ **Note:** Tkinter needs a desktop display, so this cell will only run on a **local machine** (Windows/Linux/Mac with Jupyter), **not** on Colab/Kaggle. It also expects FFmpeg at `C:\ffmpeg\bin\ffmpeg.exe` on Windows.

In [6]:
# ============================================================
# Project Title : Audio Noise Remover from Video
# Developed By  : Alok Sahni
# Language Used : Python 3.x
# Tools         : Tkinter, MoviePy, SciPy, Noisereduce, FFmpeg
# ============================================================

import os
import tkinter as tk
from tkinter import filedialog, messagebox
from tkinter.ttk import Progressbar
from moviepy.editor import VideoFileClip
from scipy.io import wavfile
import noisereduce as nr
import tempfile
import subprocess
from playsound import playsound


class NoiseRemoverApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Audio Noise Remover from Video")
        self.root.geometry("500x400")

        self.video_path = ""
        self.cleaned_audio_path = ""

        self.label = tk.Label(root, text="Select a video file (.mp4 / .mkv)",
                              font=("Arial", 12))
        self.label.pack(pady=10)

        self.select_button = tk.Button(root, text="Browse",
                                       command=self.browse_file)
        self.select_button.pack(pady=5)

        self.progress = Progressbar(root, orient=tk.HORIZONTAL,
                                    length=300, mode='determinate')
        self.progress.pack(pady=10)

        self.clean_button = tk.Button(root, text="Clean Audio",
                                      command=self.clean_audio)
        self.clean_button.pack(pady=5)

        self.preview_original_button = tk.Button(
            root, text="Preview Original Audio",
            command=self.preview_original_audio)
        self.preview_original_button.pack(pady=5)

        self.preview_cleaned_button = tk.Button(
            root, text="Preview Cleaned Audio",
            command=self.preview_cleaned_audio)
        self.preview_cleaned_button.pack(pady=5)

    def browse_file(self):
        filetypes = [("Video files", "*.mp4 *.mkv")]
        self.video_path = filedialog.askopenfilename(
            title="Open Video File", filetypes=filetypes)
        if self.video_path:
            messagebox.showinfo("Selected",
                                f"File selected:\n{self.video_path}")

    def preview_original_audio(self):
        if not self.video_path:
            messagebox.showwarning("No file",
                                   "Please select a video file first.")
            return
        temp_audio = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
        clip = VideoFileClip(self.video_path)
        clip.audio.write_audiofile(temp_audio.name, verbose=False, logger=None)
        clip.close()
        playsound(temp_audio.name)
        temp_audio.close()
        os.unlink(temp_audio.name)

    def preview_cleaned_audio(self):
        if not self.cleaned_audio_path:
            messagebox.showwarning("Not cleaned",
                                   "Please clean the audio first.")
            return
        playsound(self.cleaned_audio_path)

    def clean_audio(self):
        if not self.video_path:
            messagebox.showwarning("No file",
                                   "Please select a video file first.")
            return

        self.progress['value'] = 10
        self.root.update_idletasks()

        temp_audio_path = None
        try:
            clip = VideoFileClip(self.video_path)
            temp_audio = tempfile.NamedTemporaryFile(delete=False,
                                                     suffix=".wav")
            temp_audio_path = temp_audio.name
            temp_audio.close()
            clip.audio.write_audiofile(temp_audio_path,
                                       verbose=False, logger=None)
            clip.close()

            self.progress['value'] = 30
            self.root.update_idletasks()

            rate, data = wavfile.read(temp_audio_path)
            if len(data.shape) == 2:
                data = data.mean(axis=1)

            noise_sample = data[:5000]
            reduced_noise = nr.reduce_noise(y=data, sr=rate,
                                            y_noise=noise_sample)

            cleaned_audio_file = tempfile.NamedTemporaryFile(delete=False,
                                                             suffix=".wav")
            wavfile.write(cleaned_audio_file.name, rate,
                          reduced_noise.astype("int16"))
            self.cleaned_audio_path = cleaned_audio_file.name

            self.progress['value'] = 70
            self.root.update_idletasks()

            output_video_path = (os.path.splitext(self.video_path)[0]
                                 + "_cleaned.mp4")
            ffmpeg_path = r"C:\ffmpeg\bin\ffmpeg.exe"
            command = [
                ffmpeg_path, "-y",
                "-i", self.video_path,
                "-i", cleaned_audio_file.name,
                "-c:v", "copy",
                "-map", "0:v:0",
                "-map", "1:a:0",
                output_video_path
            ]
            result = subprocess.run(command, capture_output=True, text=True)
            if result.returncode != 0:
                print("FFmpeg Error:", result.stderr)

            self.progress['value'] = 100
            self.root.update_idletasks()

            messagebox.showinfo(
                "Success",
                f"Cleaned video saved to:\n{output_video_path}")

        except Exception as e:
            messagebox.showerror("Error", str(e))

        finally:
            if temp_audio_path and os.path.exists(temp_audio_path):
                try:
                    os.unlink(temp_audio_path)
                except Exception as cleanup_error:
                    print(f"Cleanup error: {cleanup_error}")


# Uncomment the two lines below to launch the desktop app (local machine only)
# root = tk.Tk()
# app = NoiseRemoverApp(root)
# root.mainloop()
print("NoiseRemoverApp class defined. Uncomment the last lines to launch the GUI locally.")

NoiseRemoverApp class defined. Uncomment the last lines to launch the GUI locally.


## 7. Output

- Original and cleaned audio can be previewed (in-notebook players above, or Preview buttons in the desktop app)
- Cleaned audio is saved as a `.wav` file
- Final output is a new video file (e.g., `myinterview_cleaned.mp4`) where the original noisy audio is replaced with noise-reduced audio

## 8. Conclusion

This project effectively demonstrates how Python can be used in media processing and GUI development. It bridges the gap between audio signal processing and practical usage with tools like FFmpeg. By automating the process of cleaning audio and embedding it back into the video, this tool can be a helpful utility for educators, content creators, editors, and students who want to improve the audio quality of their recorded material.

### Known Limitations
- Noise profile always taken from the first 5000 samples (assumes the recording starts with noise only)
- Desktop version uses a hard-coded Windows FFmpeg path (the notebook version auto-detects it)
- Processing is synchronous; no formal quality metrics (SNR) — verification by listening

### Bibliography (selected)
1. Python Software Foundation. (2024). *Python Official Documentation.* https://docs.python.org/3/
2. FFmpeg Developers. (2024). *FFmpeg Documentation.* https://ffmpeg.org/documentation.html
3. MoviePy Developers. (2024). *MoviePy Documentation.* https://zulko.github.io/moviepy/
4. SciPy Community. (2024). *SciPy Documentation.* https://docs.scipy.org/doc/scipy/
5. Noisereduce Developers. (2024). *noisereduce Library Documentation.* https://github.com/timsainb/noisereduce
6. Smith, J. O. (2010). *Introduction to Digital Filters with Audio Applications.* W3K Publishing.
7. Verma, T., & Sahni, A. (2025). *Audio Noise Remover from Video: A Python-Based GUI Application.* Unpublished student project.

*(Full 20-reference bibliography available in the project report.)*